# MNIST digit classification

Runs end to end on the SageMaker notebook instance:

1. download MNIST
2. upload it to the project S3 bucket (KMS encrypted)
3. train a classifier locally on this instance
4. write the model artifact back to S3

Training runs on the notebook instance itself, not a separate SageMaker
training job, so it stays inside the `ml.t3.medium` you are already paying for.

## 1. Setup

The bucket name comes from the Terraform output `data_bucket`. The execution
role is scoped to this bucket only, so pointing elsewhere will fail with
`AccessDenied`.

`conda_python3` is a minimal kernel, so scikit-learn is installed first. If you
would rather not install anything, switch the kernel to one of the preinstalled
ML images (Kernel > Change kernel) and skip the next cell.

In [3]:
%pip install --quiet scikit-learn joblib

import sklearn

print(f"scikit-learn {sklearn.__version__}")

Note: you may need to restart the kernel to use updated packages.
scikit-learn 1.7.2


In [1]:
import boto3

REGION = "ca-central-1"
PREFIX = "mnist"

# From `terraform -chdir=infra output data_bucket`.
BUCKET = "sagemaker-notebook-dev-data-099139718958"

s3 = boto3.client("s3", region_name=REGION)

# Fails loudly now rather than midway through training.
s3.head_bucket(Bucket=BUCKET)
print(f"bucket reachable: {BUCKET}")

bucket reachable: sagemaker-notebook-dev-data-099139718958


## 2. Download MNIST

`fetch_openml` pulls ~11 MB. This is the step that proves the subnet has a
route to the internet — if the notebook were on an isolated subnet it would
hang here.

In [2]:
import numpy as np
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")

X = mnist.data.astype(np.float32) / 255.0
y = mnist.target.astype(np.int64)

print(f"X: {X.shape}  y: {y.shape}")

X: (70000, 784)  y: (70000,)


## 3. Upload to S3

Saved as a single compressed `.npz`. The bucket applies `aws:kms` by default,
so this call also exercises `kms:GenerateDataKey` on the project CMK.

In [3]:
import io

buf = io.BytesIO()
np.savez_compressed(buf, X=X, y=y)
buf.seek(0)

key = f"{PREFIX}/mnist.npz"
s3.upload_fileobj(buf, BUCKET, key)

size_mb = s3.head_object(Bucket=BUCKET, Key=key)["ContentLength"] / 1024**2
print(f"s3://{BUCKET}/{key}  ({size_mb:.1f} MB)")

s3://sagemaker-notebook-dev-data-099139718958/mnist/mnist.npz  (20.1 MB)


## 4. Train

Logistic regression on a 10k subset. `ml.t3.medium` has 2 vCPU, so the full
70k rows would take a while — the subset keeps this to roughly a minute while
still landing around 90% accuracy.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

N = 10_000
X_train, X_test, y_train, y_test = train_test_split(
    X[:N], y[:N], test_size=0.2, random_state=42, stratify=y[:N]
)

# n_jobs is deprecated in sklearn >= 1.8 and has no effect on the default solver.
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

acc = accuracy_score(y_test, clf.predict(X_test))
print(f"test accuracy: {acc:.4f}")

test accuracy: 0.9010


## 5. Save the model to S3

In [5]:
import joblib

buf = io.BytesIO()
joblib.dump(clf, buf)
buf.seek(0)

model_key = f"{PREFIX}/model.joblib"
s3.upload_fileobj(buf, BUCKET, model_key)
print(f"s3://{BUCKET}/{model_key}")

s3://sagemaker-notebook-dev-data-099139718958/mnist/model.joblib


## 6. Verify the round trip

Reload the model straight from S3 and predict, confirming both read and
KMS decrypt work.

In [6]:
obj = s3.get_object(Bucket=BUCKET, Key=model_key)
reloaded = joblib.load(io.BytesIO(obj["Body"].read()))

print(f"reloaded accuracy: {accuracy_score(y_test, reloaded.predict(X_test)):.4f}")

for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX).get("Contents", []):
    print(f"  {o['Key']}  {o['Size'] / 1024**2:.1f} MB")

reloaded accuracy: 0.9010
  mnist/mnist.npz  20.1 MB
  mnist/model.joblib  0.1 MB
